# 08 - Warmup and Learning-Rate Schedules

This notebook isolates the learning-rate machinery: the `Warmup` ramp and the
`Scheduler` decay policies. We drive each component over a synthetic step /
epoch sequence and plot the resulting learning-rate multiplier so the shape
of every schedule can be verified by eye.

Modules exercised: `pipelines.training_pipeline.callbacks.Warmup`,
`pipelines.training_pipeline.callbacks.Scheduler`,
`configuration.training_config.WarmupConfig` / `SchedulerConfig`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED   = 0
DEVICE = torch.device("cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "savefig.dpi"    : 300,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
})

print("repo root :", REPO_ROOT)
print("torch     :", torch.__version__)
print("device    :", DEVICE)


In [ ]:
import copy
from configuration.training_config import TrainerConfig, GaussianConfig, WarmupConfig
from pipelines.training_pipeline.callbacks import Warmup, Scheduler
from tools import NullLogger, NullTracker

gaussian_cfg = GaussianConfig(n_default_gaussians=2, x_min=-10.0, x_max=40.0)
base_cfg     = TrainerConfig(gaussian=gaussian_cfg)
nl, nt       = NullLogger(), NullTracker()


## Warmup ramp modes

The warmup factor rises from `warmup_start_factor` to 1.0 over
`warmup_steps`. We trace the factor step-by-step for each supported mode.

In [ ]:
WARMUP_STEPS = 100
modes        = ['linear', 'cosine', 'exponential', 'polynomial']

fig, ax = plt.subplots(figsize=(6.5, 4))
for mode in modes:
    cfg = copy.deepcopy(base_cfg)
    cfg.warmup = WarmupConfig(warmup_steps=WARMUP_STEPS, warmup_start_factor=0.1,
                              warmup_enabled=True, warmup_mode=mode, warmup_poly_power=2.0)
    w = Warmup(cfg, nl, nt)
    factors = [w.step() for _ in range(WARMUP_STEPS + 20)]
    ax.plot(factors, label=mode)

ax.axvline(WARMUP_STEPS, color='grey', ls=':', lw=1, label='warmup end')
ax.set_xlabel('optimizer step')
ax.set_ylabel('warmup factor')
ax.set_title('Warmup ramp modes')
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## Scheduler decay policies (no warmup)

Each scheduler returns a per-epoch learning rate from a fixed base LR. We
build a scheduler per policy with `warmup=None` and trace the LR over epochs.
For `reduce_on_plateau` we feed a synthetic validation metric that stops
improving, so the policy fires its decay after the patience window.

In [ ]:
BASE_LR = 1e-3
EPOCHS  = 100
policies = ['cosine_annealing', 'step', 'multi_step', 'exponential',
            'linear', 'polynomial', 'constant', 'cosine_annealing_warm_restarts']

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for pol in policies:
    cfg = copy.deepcopy(base_cfg)
    cfg.scheduler.type   = pol
    cfg.scheduler.epochs = EPOCHS
    cfg.scheduler.T_0    = 25
    sch = Scheduler([BASE_LR], None, cfg, nl, nt)
    lrs = [sch.step(e)[0] for e in range(EPOCHS)]
    ax.plot(lrs, label=pol)

ax.set_xlabel('epoch')
ax.set_ylabel('learning rate')
ax.set_title('Scheduler decay policies')
ax.legend(frameon=False, fontsize=8, ncol=2)
fig.tight_layout()
plt.show()


In [ ]:
cfg = copy.deepcopy(base_cfg)
cfg.scheduler.type      = 'reduce_on_plateau'
cfg.scheduler.factor    = 0.5
cfg.scheduler.patience  = 5
cfg.scheduler.threshold = 1e-4
sch = Scheduler([BASE_LR], None, cfg, nl, nt)

metrics = [1.0 / (1.0 + e) if e < 15 else 0.05 for e in range(EPOCHS)]
lrs = [sch.step(e, metric=metrics[e])[0] for e in range(EPOCHS)]

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(lrs, color='C0', label='learning rate')
ax1.set_xlabel('epoch'); ax1.set_ylabel('learning rate', color='C0')
ax2 = ax1.twinx()
ax2.plot(metrics, color='C3', ls='--', label='val metric')
ax2.set_ylabel('val metric', color='C3'); ax2.grid(False)
ax1.set_title('reduce_on_plateau response to a plateauing metric')
fig.tight_layout()
plt.show()


## Expected visual outcome

The warmup panel shows four monotone ramps from 0.1 to 1.0 reaching the
plateau at the warmup-end line, differing only in curvature. The decay panel
shows the characteristic shape of each policy: smooth cosine decay, staircase
step/multi-step drops, exponential and polynomial decay, a flat constant
line, and sawtooth warm restarts. The plateau panel shows the LR holding
while the metric improves, then dropping by the configured factor once the
metric stalls beyond the patience window.